# D10 · J1939, ISO-TP and UDS diagnostics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/02_drivers_protocols/D10_j1939_iso_tp_and_uds_diagnostics.ipynb)

**Track:** Drivers and protocols · **Skill axis (sheet):** Drivers and protocols · **Time:** about 65 min

**Skill Hub concepts:** J1939 parameter group numbers; J1939 address claiming; ISO-TP transport segmentation; Unified Diagnostic Services

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("D10")

## Map: heavy-vehicle CAN and the diagnostic stack

**J1939 29-bit identifier**

```
 28..26     25    24   23..16   15..8   7..0
 priority | EDP | DP |   PF   |   PS   |  SA
 PF < 240 (PDU1): PS = destination address, PGN = (EDP, DP, PF) << 8          (PS not part of the PGN)
 PF >= 240 (PDU2): PS = group extension,    PGN = (EDP, DP, PF, PS)           (broadcast)
```

Address claiming (PGN 0xEE00): each node broadcasts its 64-bit NAME for the address it wants; on a conflict the
**lower NAME keeps the address**, the other picks another one or sends "cannot claim".

**ISO-TP (ISO 15765-2)** carries messages longer than one CAN frame:

| Frame | First nibble | Layout (classic CAN, 8 bytes) |
|---|---|---|
| Single | 0 | `0L` + up to 7 bytes |
| First | 1 | `1L LL` (12-bit length) + 6 bytes |
| Consecutive | 2 | `2N` (sequence number 1, 2, ... 15, 0, 1 ...) + 7 bytes |
| Flow control | 3 | `30 BS STmin` from the receiver: block size, minimum gap |

**UDS (ISO 14229)** on top of ISO-TP:

| Service | ID | Positive response |
|---|---|---|
| DiagnosticSessionControl | 0x10 | 0x50 |
| ECUReset | 0x11 | 0x51 |
| ReadDataByIdentifier | 0x22 | 0x62 |
| SecurityAccess | 0x27 | 0x67 |
| WriteDataByIdentifier | 0x2E | 0x6E |
| RoutineControl | 0x31 | 0x71 |
| RequestDownload / TransferData / RequestTransferExit | 0x34 / 0x36 / 0x37 | +0x40 |
| TesterPresent | 0x3E | 0x7E |

Negative response: `7F <SID> <NRC>`; NRC 0x11 service not supported, 0x13 incorrect length, 0x31 request out of
range, 0x33 security access denied, 0x78 response pending, 0x7F service not supported in active session.

## Task 1: parse J1939 identifiers and settle an address claim

In [ ]:
def parse_j1939(can_id):
    """Return dict(priority, pgn, sa, da) with da None for PDU2 (broadcast)."""
    raise NotImplementedError  # TODO

def claim_winner(names):
    raise NotImplementedError  # TODO

lab.check("j1939", parse_j1939, claim_winner)

In [ ]:
%%writefile d10_j1939.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/d10.h
// Grade it with lab.check("j1939", "d10_j1939.c"). For C++, name the file .cpp in both places.
#include "d10.h"

void parse_j1939(uint32_t can_id, J1939Id *out) {
    (void)can_id;
    out->priority = 0; out->pgn = 0; out->sa = 0; out->da = -1; /* TODO: PDU1 if PF < 240 */
}

uint64_t claim_winner(const uint64_t *names, size_t n) { (void)names; (void)n; return 0; /* TODO */ }

In [ ]:
# lab.check("j1939", "d10_j1939.c")  # uncomment after writing the C/C++ cell above

## Task 2: ISO-TP segmentation and reassembly (classic CAN, pad unused bytes with 0xCC)

In [ ]:
def isotp_segment(payload):
    raise NotImplementedError  # TODO

def isotp_reassemble(frames):
    raise NotImplementedError  # TODO

lab.check("isotp", isotp_segment, isotp_reassemble)

In [ ]:
%%writefile d10_isotp.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/d10.h
// Grade it with lab.check("isotp", "d10_isotp.c"). For C++, name the file .cpp in both places.
#include "d10.h"

size_t isotp_segment(const uint8_t *payload, size_t n, uint8_t *frames, size_t max_frames) {
    (void)payload; (void)n; (void)frames; (void)max_frames;
    return 0; /* TODO: single frame up to 7 bytes, else first frame + consecutive frames, pad with 0xCC */
}

int isotp_reassemble(const uint8_t *frames, size_t n_frames, uint8_t *out, size_t out_cap) {
    (void)frames; (void)n_frames; (void)out; (void)out_cap;
    return 0; /* TODO */
}

In [ ]:
# lab.check("isotp", "d10_isotp.c")  # uncomment after writing the C/C++ cell above

## Task 3: a tiny UDS server
Support: 0x10 (sub 0x01 default, 0x03 extended), 0x22 ReadDataByIdentifier, 0x2E WriteDataByIdentifier (extended
session only, else NRC 0x7F). Unknown service: NRC 0x11. Wrong length: NRC 0x13. Unknown DID: NRC 0x31.
Positive response to 0x10 is `50 <sub>`, to 0x22 `62 <DID hi> <DID lo> <data>`, to 0x2E `6E <DID hi> <DID lo>`.

In [ ]:
class UdsServer:
    def __init__(self):
        self.session = 0x01
        self.dids = {0xF190: b"BZ-STORM-0000001", 0xF187: b"STM32G474"}

    def handle(self, req: bytes) -> bytes:
        raise NotImplementedError  # TODO

lab.check("uds", UdsServer)

In [ ]:
%%writefile d10_uds.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/d10.h
// Grade it with lab.check("uds", "d10_uds.c"). For C++, name the file .cpp in both places.
#include "d10.h"

void uds_init(UdsServer *s) {
    s->session = 0x01;
    s->n_dids = 0; /* TODO: load 0xF190 and 0xF187 */
}

size_t uds_handle(UdsServer *s, const uint8_t *req, size_t len, uint8_t *resp, size_t resp_cap) {
    (void)s; (void)req; (void)len; (void)resp; (void)resp_cap;
    return 0; /* TODO: 0x10 session control, 0x22 read DID, 0x2E write DID (extended session), else NRC 0x11 */
}

In [ ]:
# lab.check("uds", "d10_uds.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** PGN carried by identifier `0x0CF00400` (decimal)?

**M2 (Apply).** ISO-TP frames (excluding flow control) to send a 100-byte message over classic CAN?

**M3 (Remember).** Positive response SID for ReadDataByIdentifier (decimal)?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Why does ISO-TP have a flow control frame?
- a) to carry the CRC
- b) the receiver sets block size and minimum separation time, so a slow ECU's buffer and CPU are not overrun
- c) to request retransmission of lost frames

**K2 (Analyze).** Two identical J1939 controllers boot and both claim address 0x80. Walk through what happens on the
bus and how the NAME fields make the outcome deterministic.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `d10.h`).
3. In `roboprep/checks/d10.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()